# Pathumma Vision 3.0 resolution sweep: raw versus gray220

Full 21-page OCR output is retained in this notebook for every run. The matrix is six longest-edge settings (`1024`, `1400`, `1600`, `1800`, `2000`, `2200`) crossed with original raw images and `gray220` images. Per-page preprocessing audit, GPU snapshots, benchmark configuration, timing, and Golden scores are captured as cell output.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import time
import urllib.request

ROOT = Path(".")
AUDIT_PYTHON = Path("python")
GOLDEN = ROOT / "results/bot_credit_bureau_21p_golden_transcript_20260829.json"
SWEEP_ROOT = ROOT / "inputs/bot_credit_bureau_2559/resolution_sweep"
MANIFEST = SWEEP_ROOT / "gray220_resolution_sweep_audit.json"
MODEL = "pathumma-vision-3-0-0-preview"
ENDPOINT = "http://127.0.0.1:8093"
PROMPT_PROFILE = "document"
CONCURRENCY = 7
SERVER_MAX_NUM_SEQS = 7
MTP_TOKENS = 0
DISABLE_THINKING = False
SIZES = (1024, 1400, 1600, 1800, 2000, 2200)
TEMPERATURE = 0.0
TOP_P = 0.8
TOP_K = 20
REPETITION_PENALTY = 1.05
PRESENCE_PENALTY = 0.0
MAX_TOKENS = 8192

def gpu_snapshot():
    return subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.used", "--format=csv,noheader,nounits"],
        text=True,
    ).strip()

def wait_for_model(timeout_seconds=900):
    deadline = time.monotonic() + timeout_seconds
    while time.monotonic() < deadline:
        try:
            with urllib.request.urlopen(ENDPOINT + "/v1/models", timeout=5) as response:
                payload = json.load(response)
            print("Endpoint ready:", ", ".join(item["id"] for item in payload.get("data", [])))
            return
        except Exception as error:
            last_error = error
            time.sleep(5)
    raise TimeoutError(f"Endpoint did not become ready: {last_error}")

def run_ocr(label, image_directory, result_path):
    output = ROOT / result_path
    if output.exists():
        raise FileExistsError(f"Refusing to overwrite {output}")
    command = [
        str(AUDIT_PYTHON), "benchmark/ocr_benchmark.py", str(ROOT / image_directory), str(output),
        "--endpoint", ENDPOINT,
        "--model", MODEL,
        "--prompt-profile", PROMPT_PROFILE,
        "--temperature", str(TEMPERATURE),
        "--top-p", str(TOP_P),
        "--top-k", str(TOP_K),
        "--repetition-penalty", str(REPETITION_PENALTY),
        "--presence-penalty", str(PRESENCE_PENALTY),
        "--max-tokens", str(MAX_TOKENS),
        "--concurrency", str(CONCURRENCY),
        "--images-per-request", "1",
        "--server-max-num-seqs", str(SERVER_MAX_NUM_SEQS),
    ]
    if DISABLE_THINKING:
        command.append("--disable-thinking")
    print(f"\n=== {label} ===")
    print("GPU before:\n" + gpu_snapshot())
    print("Command:", " ".join(command))
    completed = subprocess.run(command, cwd=ROOT, text=True, capture_output=True, check=False)
    print(completed.stdout)
    if completed.stderr:
        print("STDERR:\n" + completed.stderr)
    print("GPU after:\n" + gpu_snapshot())
    if completed.returncode:
        raise RuntimeError(f"OCR exited with {completed.returncode}")
    return output

def show_all_pages(result_path):
    payload = json.loads(Path(result_path).read_text(encoding="utf-8"))
    print(json.dumps({"configuration": payload["configuration"], "summary": payload["summary"]}, ensure_ascii=False, indent=2))
    records = sorted(payload["records"], key=lambda item: int(Path(item["images"][0]).stem.split("-")[-1]))
    for record in records:
        page = int(Path(record["images"][0]).stem.split("-")[-1])
        print(f"\n--- OCR output: page {page:02d} ({record['elapsed_seconds']}s) ---")
        print(record.get("text", ""))

def show_gray_audit(longest_edge):
    payload = json.loads(MANIFEST.read_text(encoding="utf-8"))
    records = payload["sizes"][str(longest_edge)]["records"]
    print(json.dumps(records, ensure_ascii=False, indent=2))


In [2]:
sys.path.insert(0, str(ROOT / "benchmark"))
from ocr_benchmark import PROMPTS

print("Prompt profile:", PROMPT_PROFILE)
print(PROMPTS[PROMPT_PROFILE])
print("\nRun controls:")
print(json.dumps({
    "model": MODEL,
    "endpoint": ENDPOINT,
    "MTP_tokens": MTP_TOKENS,
    "server_max_num_seqs": SERVER_MAX_NUM_SEQS,
    "concurrency": CONCURRENCY,
    "disable_thinking": DISABLE_THINKING,
    "temperature": TEMPERATURE,
    "max_tokens": MAX_TOKENS,
}, ensure_ascii=False, indent=2))
wait_for_model()


Prompt profile: document
อ่านข้อความทุกส่วนของหน้ากระดาษนี้อย่างถูกต้องที่สุด และส่งคืนเฉพาะ Markdown ที่สกัดได้

กติกา:
- เก็บข้อความภาษาไทย อังกฤษ ตัวเลขอารบิก และตัวเลขไทยตามภาพ ห้ามแก้ไขหรือแปลงเลขไทยเป็นเลขอารบิก
- เก็บลำดับการอ่าน หัวข้อ รายการย่อย หัว/ท้ายกระดาษ และเชิงอรรถที่มองเห็น
- ตารางให้ส่งเป็น HTML <table> โดยคงแถว คอลัมน์ และหัวตาราง; ห้ามสรุปหรือข้ามเซลล์
- กราฟ แผนภาพ workflow และรูป ให้ห่อด้วย <figure> พร้อมถอดข้อความทุกคำก่อน แล้วบรรยายองค์ประกอบเท่าที่เห็น
- เมื่ออ่านข้อความไม่ชัด ให้เขียน [อ่านไม่ชัด] เฉพาะตำแหน่งนั้น ห้ามเดาหรือใช้ความรู้นอกภาพ
- ห้ามอธิบายวิธีทำ ห้ามเพิ่มข้อมูลที่ไม่อยู่ในภาพ

Run controls:
{
  "model": "pathumma-vision-3-0-0-preview",
  "endpoint": "http://127.0.0.1:8093",
  "MTP_tokens": 0,
  "server_max_num_seqs": 7,
  "concurrency": 7,
  "disable_thinking": false,
  "temperature": 0.0,
  "max_tokens": 8192
}
Endpoint ready: pathumma-vision-3-0-0-preview


## 1024px raw

In [3]:
raw_1024 = run_ocr("raw 1024px", "inputs/bot_credit_bureau_2559/resolution_sweep/raw_1024", "results/pathumma_bot_credit_bureau_21p_raw_px1024_resolution_sweep_20260829.json")
show_all_pages(raw_1024)



=== raw 1024px ===
GPU before:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 11586
Command: python benchmark/ocr_benchmark.py ./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1024 ./results/pathumma_bot_credit_bureau_21p_raw_px1024_resolution_sweep_20260829.json --endpoint http://127.0.0.1:8093 --model pathumma-vision-3-0-0-preview --prompt-profile document --temperature 0.0 --top-p 0.8 --top-k 20 --repetition-penalty 1.05 --presence-penalty 0.0 --max-tokens 8192 --concurrency 7 --images-per-request 1 --server-max-num-seqs 7


completed request 3/21 (pages 003)
completed request 6/21 (pages 006)
completed request 7/21 (pages 007)
completed request 9/21 (pages 009)
completed request 10/21 (pages 010)
completed request 11/21 (pages 011)
completed request 12/21 (pages 012)
completed request 14/21 (pages 014)
completed request 15/21 (pages 015)
completed request 16/21 (pages 016)
completed request 1/21 (pages 001)
completed request 2/21 (pages 002)
completed request 4/21 (pages 004)
completed request 5/21 (pages 005)
completed request 18/21 (pages 018)
completed request 17/21 (pages 017)
completed request 20/21 (pages 020)
completed request 8/21 (pages 008)
completed request 19/21 (pages 019)
completed request 13/21 (pages 013)
completed request 21/21 (pages 021)
{"images": 21, "requests": 21, "elapsed_seconds": 86.9479, "seconds_per_image": 4.1404, "completion_tokens": 75157, "end_to_end_completion_tokens_per_second": 864.391}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 12283
{
  "c

## 1024px gray220

In [4]:
show_gray_audit(1024)
gray_1024 = run_ocr("gray220 1024px", "inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1024", "results/pathumma_bot_credit_bureau_21p_gray220_px1024_resolution_sweep_20260829.json")
show_all_pages(gray_1024)


[
  {
    "page": 1,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-01.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1024/page-01.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1024/page-01.png",
    "dimensions": [
      724,
      1024
    ],
    "threshold": 220,
    "pixels_replaced": 701722,
    "pixels_total": 741376,
    "pixels_replaced_percent": 94.6513,
    "raw_sha256": "52d5568aaef582dcec78889ffe4cf949683dd20c55c73ad0efb135690563596e",
    "gray220_sha256": "361f82d8dcccbf24b406e0de1d588675d31b4ba41964aeb8dddd18dc65ee73a3"
  },
  {
    "page": 2,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-02.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1024/page-02.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1024/page-02.png",
    "dimensions": [
      724,
      1024
    ],
    "threshold": 220,
    "pi

completed request 1/21 (pages 001)
completed request 5/21 (pages 005)
completed request 4/21 (pages 004)
completed request 3/21 (pages 003)
completed request 6/21 (pages 006)
completed request 2/21 (pages 002)
completed request 7/21 (pages 007)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 10/21 (pages 010)
completed request 13/21 (pages 013)
completed request 14/21 (pages 014)
completed request 11/21 (pages 011)
completed request 12/21 (pages 012)
completed request 21/21 (pages 021)
completed request 18/21 (pages 018)
completed request 16/21 (pages 016)
completed request 15/21 (pages 015)
completed request 17/21 (pages 017)
completed request 20/21 (pages 020)
completed request 19/21 (pages 019)
{"images": 21, "requests": 21, "elapsed_seconds": 20.2836, "seconds_per_image": 0.9659, "completion_tokens": 21182, "end_to_end_completion_tokens_per_second": 1044.294}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 12279
{
  "

## 1400px raw

In [5]:
raw_1400 = run_ocr("raw 1400px", "inputs/bot_credit_bureau_2559/resolution_sweep/raw_1400", "results/pathumma_bot_credit_bureau_21p_raw_px1400_resolution_sweep_20260829.json")
show_all_pages(raw_1400)



=== raw 1400px ===
GPU before:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 12279
Command: python benchmark/ocr_benchmark.py ./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1400 ./results/pathumma_bot_credit_bureau_21p_raw_px1400_resolution_sweep_20260829.json --endpoint http://127.0.0.1:8093 --model pathumma-vision-3-0-0-preview --prompt-profile document --temperature 0.0 --top-p 0.8 --top-k 20 --repetition-penalty 1.05 --presence-penalty 0.0 --max-tokens 8192 --concurrency 7 --images-per-request 1 --server-max-num-seqs 7


completed request 1/21 (pages 001)
completed request 4/21 (pages 004)
completed request 5/21 (pages 005)
completed request 3/21 (pages 003)
completed request 6/21 (pages 006)
completed request 2/21 (pages 002)
completed request 7/21 (pages 007)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 11/21 (pages 011)
completed request 13/21 (pages 013)
completed request 14/21 (pages 014)
completed request 12/21 (pages 012)
completed request 10/21 (pages 010)
completed request 16/21 (pages 016)
completed request 17/21 (pages 017)
completed request 20/21 (pages 020)
completed request 15/21 (pages 015)
completed request 19/21 (pages 019)
completed request 18/21 (pages 018)
completed request 21/21 (pages 021)
{"images": 21, "requests": 21, "elapsed_seconds": 49.8233, "seconds_per_image": 2.3725, "completion_tokens": 38425, "end_to_end_completion_tokens_per_second": 771.226}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 12268
{
  "c

## 1400px gray220

In [6]:
show_gray_audit(1400)
gray_1400 = run_ocr("gray220 1400px", "inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1400", "results/pathumma_bot_credit_bureau_21p_gray220_px1400_resolution_sweep_20260829.json")
show_all_pages(gray_1400)


[
  {
    "page": 1,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-01.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1400/page-01.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1400/page-01.png",
    "dimensions": [
      990,
      1400
    ],
    "threshold": 220,
    "pixels_replaced": 1323612,
    "pixels_total": 1386000,
    "pixels_replaced_percent": 95.4987,
    "raw_sha256": "027ea1c68561df52ce2beea388ee8c612768006d6feb1f4fa09bcabcff165a18",
    "gray220_sha256": "48c9be1e4a099ded0fdef40c00daade1db540a7efd6315a691070fb821bf077c"
  },
  {
    "page": 2,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-02.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1400/page-02.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1400/page-02.png",
    "dimensions": [
      990,
      1400
    ],
    "threshold": 220,
    "

completed request 1/21 (pages 001)
completed request 5/21 (pages 005)
completed request 4/21 (pages 004)
completed request 3/21 (pages 003)
completed request 6/21 (pages 006)
completed request 2/21 (pages 002)
completed request 7/21 (pages 007)
completed request 9/21 (pages 009)
completed request 8/21 (pages 008)
completed request 10/21 (pages 010)
completed request 14/21 (pages 014)
completed request 13/21 (pages 013)
completed request 11/21 (pages 011)
completed request 12/21 (pages 012)
completed request 21/21 (pages 021)
completed request 18/21 (pages 018)
completed request 16/21 (pages 016)
completed request 15/21 (pages 015)
completed request 17/21 (pages 017)
completed request 20/21 (pages 020)
completed request 19/21 (pages 019)
{"images": 21, "requests": 21, "elapsed_seconds": 20.9945, "seconds_per_image": 0.9997, "completion_tokens": 21169, "end_to_end_completion_tokens_per_second": 1008.311}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 12279
{
  "

## 1600px raw

In [7]:
raw_1600 = run_ocr("raw 1600px", "inputs/bot_credit_bureau_2559/resolution_sweep/raw_1600", "results/pathumma_bot_credit_bureau_21p_raw_px1600_resolution_sweep_20260829.json")
show_all_pages(raw_1600)



=== raw 1600px ===
GPU before:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 12279
Command: python benchmark/ocr_benchmark.py ./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1600 ./results/pathumma_bot_credit_bureau_21p_raw_px1600_resolution_sweep_20260829.json --endpoint http://127.0.0.1:8093 --model pathumma-vision-3-0-0-preview --prompt-profile document --temperature 0.0 --top-p 0.8 --top-k 20 --repetition-penalty 1.05 --presence-penalty 0.0 --max-tokens 8192 --concurrency 7 --images-per-request 1 --server-max-num-seqs 7


completed request 5/21 (pages 005)
completed request 3/21 (pages 003)
completed request 6/21 (pages 006)
completed request 2/21 (pages 002)
completed request 7/21 (pages 007)
completed request 8/21 (pages 008)
completed request 10/21 (pages 010)
completed request 11/21 (pages 011)
completed request 12/21 (pages 012)
completed request 9/21 (pages 009)
completed request 14/21 (pages 014)
completed request 16/21 (pages 016)
completed request 15/21 (pages 015)
completed request 17/21 (pages 017)
completed request 20/21 (pages 020)
completed request 19/21 (pages 019)
completed request 1/21 (pages 001)
completed request 4/21 (pages 004)
completed request 13/21 (pages 013)
completed request 18/21 (pages 018)
completed request 21/21 (pages 021)
{"images": 21, "requests": 21, "elapsed_seconds": 67.5219, "seconds_per_image": 3.2153, "completion_tokens": 61107, "end_to_end_completion_tokens_per_second": 904.995}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 12601
{
  "c

## 1600px gray220

In [8]:
show_gray_audit(1600)
gray_1600 = run_ocr("gray220 1600px", "inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1600", "results/pathumma_bot_credit_bureau_21p_gray220_px1600_resolution_sweep_20260829.json")
show_all_pages(gray_1600)


[
  {
    "page": 1,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-01.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1600/page-01.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1600/page-01.png",
    "dimensions": [
      1131,
      1600
    ],
    "threshold": 220,
    "pixels_replaced": 1732917,
    "pixels_total": 1809600,
    "pixels_replaced_percent": 95.7624,
    "raw_sha256": "7cd88e5cf12e59d83931179971c152df49e476e3e08d7ba1eb6be1fa0f5d46b7",
    "gray220_sha256": "eaf2a4fa6c8aa8ea8200b12cd7eaaa81dd4aa418aab469cf58f59cbc1d5c13e2"
  },
  {
    "page": 2,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-02.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1600/page-02.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1600/page-02.png",
    "dimensions": [
      1131,
      1600
    ],
    "threshold": 220,
   

completed request 1/21 (pages 001)
completed request 5/21 (pages 005)
completed request 4/21 (pages 004)
completed request 3/21 (pages 003)
completed request 6/21 (pages 006)
completed request 2/21 (pages 002)
completed request 7/21 (pages 007)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 10/21 (pages 010)
completed request 13/21 (pages 013)
completed request 14/21 (pages 014)
completed request 11/21 (pages 011)
completed request 12/21 (pages 012)
completed request 21/21 (pages 021)
completed request 18/21 (pages 018)
completed request 16/21 (pages 016)
completed request 15/21 (pages 015)
completed request 17/21 (pages 017)
completed request 20/21 (pages 020)
completed request 19/21 (pages 019)
{"images": 21, "requests": 21, "elapsed_seconds": 19.0931, "seconds_per_image": 0.9092, "completion_tokens": 21160, "end_to_end_completion_tokens_per_second": 1108.256}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 12303
{
  "

## 1800px raw

In [9]:
raw_1800 = run_ocr("raw 1800px", "inputs/bot_credit_bureau_2559/resolution_sweep/raw_1800", "results/pathumma_bot_credit_bureau_21p_raw_px1800_resolution_sweep_20260829.json")
show_all_pages(raw_1800)



=== raw 1800px ===
GPU before:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 12303
Command: python benchmark/ocr_benchmark.py ./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1800 ./results/pathumma_bot_credit_bureau_21p_raw_px1800_resolution_sweep_20260829.json --endpoint http://127.0.0.1:8093 --model pathumma-vision-3-0-0-preview --prompt-profile document --temperature 0.0 --top-p 0.8 --top-k 20 --repetition-penalty 1.05 --presence-penalty 0.0 --max-tokens 8192 --concurrency 7 --images-per-request 1 --server-max-num-seqs 7


completed request 5/21 (pages 005)
completed request 4/21 (pages 004)
completed request 7/21 (pages 007)
completed request 6/21 (pages 006)
completed request 2/21 (pages 002)
completed request 3/21 (pages 003)
completed request 8/21 (pages 008)
completed request 11/21 (pages 011)
completed request 9/21 (pages 009)
completed request 10/21 (pages 010)
completed request 12/21 (pages 012)
completed request 14/21 (pages 014)
completed request 15/21 (pages 015)
completed request 17/21 (pages 017)
completed request 19/21 (pages 019)
completed request 20/21 (pages 020)
completed request 1/21 (pages 001)
completed request 13/21 (pages 013)
completed request 16/21 (pages 016)
completed request 18/21 (pages 018)
completed request 21/21 (pages 021)
{"images": 21, "requests": 21, "elapsed_seconds": 65.9029, "seconds_per_image": 3.1382, "completion_tokens": 61607, "end_to_end_completion_tokens_per_second": 934.815}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 12266
{
  "c

## 1800px gray220

In [10]:
show_gray_audit(1800)
gray_1800 = run_ocr("gray220 1800px", "inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1800", "results/pathumma_bot_credit_bureau_21p_gray220_px1800_resolution_sweep_20260829.json")
show_all_pages(gray_1800)


[
  {
    "page": 1,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-01.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1800/page-01.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1800/page-01.png",
    "dimensions": [
      1272,
      1800
    ],
    "threshold": 220,
    "pixels_replaced": 2197434,
    "pixels_total": 2289600,
    "pixels_replaced_percent": 95.9746,
    "raw_sha256": "ea45e6faa8ac8d1bc413f017d0428dae7458faee05babfde03c571c008faf256",
    "gray220_sha256": "e9aaa8c9c0ebf36bd4f65e716f6e8ebe06fa133de12a32db5f5af0df5df13234"
  },
  {
    "page": 2,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-02.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1800/page-02.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1800/page-02.png",
    "dimensions": [
      1272,
      1800
    ],
    "threshold": 220,
   

completed request 1/21 (pages 001)
completed request 5/21 (pages 005)
completed request 4/21 (pages 004)
completed request 3/21 (pages 003)
completed request 2/21 (pages 002)
completed request 6/21 (pages 006)
completed request 7/21 (pages 007)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 10/21 (pages 010)
completed request 13/21 (pages 013)
completed request 14/21 (pages 014)
completed request 11/21 (pages 011)
completed request 12/21 (pages 012)
completed request 21/21 (pages 021)
completed request 18/21 (pages 018)
completed request 16/21 (pages 016)
completed request 17/21 (pages 017)
completed request 15/21 (pages 015)
completed request 20/21 (pages 020)
completed request 19/21 (pages 019)
{"images": 21, "requests": 21, "elapsed_seconds": 20.9044, "seconds_per_image": 0.9954, "completion_tokens": 21150, "end_to_end_completion_tokens_per_second": 1011.749}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 12274
{
  "

## 2000px raw

In [11]:
raw_2000 = run_ocr("raw 2000px", "inputs/bot_credit_bureau_2559/resolution_sweep/raw_2000", "results/pathumma_bot_credit_bureau_21p_raw_px2000_resolution_sweep_20260829.json")
show_all_pages(raw_2000)



=== raw 2000px ===
GPU before:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 12274
Command: python benchmark/ocr_benchmark.py ./inputs/bot_credit_bureau_2559/resolution_sweep/raw_2000 ./results/pathumma_bot_credit_bureau_21p_raw_px2000_resolution_sweep_20260829.json --endpoint http://127.0.0.1:8093 --model pathumma-vision-3-0-0-preview --prompt-profile document --temperature 0.0 --top-p 0.8 --top-k 20 --repetition-penalty 1.05 --presence-penalty 0.0 --max-tokens 8192 --concurrency 7 --images-per-request 1 --server-max-num-seqs 7


completed request 1/21 (pages 001)
completed request 2/21 (pages 002)
completed request 3/21 (pages 003)
completed request 7/21 (pages 007)
completed request 6/21 (pages 006)
completed request 8/21 (pages 008)
completed request 10/21 (pages 010)
completed request 11/21 (pages 011)
completed request 12/21 (pages 012)
completed request 13/21 (pages 013)
completed request 14/21 (pages 014)
completed request 15/21 (pages 015)
completed request 17/21 (pages 017)
completed request 19/21 (pages 019)
completed request 20/21 (pages 020)
completed request 4/21 (pages 004)
completed request 5/21 (pages 005)
completed request 9/21 (pages 009)
completed request 16/21 (pages 016)
completed request 18/21 (pages 018)
completed request 21/21 (pages 021)
{"images": 21, "requests": 21, "elapsed_seconds": 69.4877, "seconds_per_image": 3.3089, "completion_tokens": 68318, "end_to_end_completion_tokens_per_second": 983.166}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 12292
{
  "c

## 2000px gray220

In [12]:
show_gray_audit(2000)
gray_2000 = run_ocr("gray220 2000px", "inputs/bot_credit_bureau_2559/resolution_sweep/gray220_2000", "results/pathumma_bot_credit_bureau_21p_gray220_px2000_resolution_sweep_20260829.json")
show_all_pages(gray_2000)


[
  {
    "page": 1,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-01.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_2000/page-01.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_2000/page-01.png",
    "dimensions": [
      1414,
      2000
    ],
    "threshold": 220,
    "pixels_replaced": 2718321,
    "pixels_total": 2828000,
    "pixels_replaced_percent": 96.1217,
    "raw_sha256": "d4a80b77100cb787878ee295a47b493e6c6afbe6b6619cf11bad88dc30e67a46",
    "gray220_sha256": "ee2902a15cdec3354762e8361c56bb13eb98bf8e0675d74bf0f719754b0f9ae3"
  },
  {
    "page": 2,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-02.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_2000/page-02.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_2000/page-02.png",
    "dimensions": [
      1414,
      2000
    ],
    "threshold": 220,
   

completed request 1/21 (pages 001)
completed request 5/21 (pages 005)
completed request 4/21 (pages 004)
completed request 6/21 (pages 006)
completed request 3/21 (pages 003)
completed request 2/21 (pages 002)
completed request 7/21 (pages 007)
completed request 11/21 (pages 011)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 10/21 (pages 010)
completed request 13/21 (pages 013)
completed request 14/21 (pages 014)
completed request 12/21 (pages 012)
completed request 21/21 (pages 021)
completed request 18/21 (pages 018)
completed request 16/21 (pages 016)
completed request 17/21 (pages 017)
completed request 15/21 (pages 015)
completed request 20/21 (pages 020)
completed request 19/21 (pages 019)
{"images": 21, "requests": 21, "elapsed_seconds": 20.8998, "seconds_per_image": 0.9952, "completion_tokens": 20805, "end_to_end_completion_tokens_per_second": 995.465}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 12398
{
  "c

## 2200px raw

In [13]:
raw_2200 = run_ocr("raw 2200px", "inputs/bot_credit_bureau_2559/resolution_sweep/raw_2200", "results/pathumma_bot_credit_bureau_21p_raw_px2200_resolution_sweep_20260829.json")
show_all_pages(raw_2200)



=== raw 2200px ===
GPU before:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 12398
Command: python benchmark/ocr_benchmark.py ./inputs/bot_credit_bureau_2559/resolution_sweep/raw_2200 ./results/pathumma_bot_credit_bureau_21p_raw_px2200_resolution_sweep_20260829.json --endpoint http://127.0.0.1:8093 --model pathumma-vision-3-0-0-preview --prompt-profile document --temperature 0.0 --top-p 0.8 --top-k 20 --repetition-penalty 1.05 --presence-penalty 0.0 --max-tokens 8192 --concurrency 7 --images-per-request 1 --server-max-num-seqs 7


completed request 6/21 (pages 006)
completed request 7/21 (pages 007)
completed request 3/21 (pages 003)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 10/21 (pages 010)
completed request 11/21 (pages 011)
completed request 12/21 (pages 012)
completed request 15/21 (pages 015)
completed request 16/21 (pages 016)
completed request 1/21 (pages 001)
completed request 2/21 (pages 002)
completed request 17/21 (pages 017)
completed request 4/21 (pages 004)
completed request 5/21 (pages 005)
completed request 19/21 (pages 019)
completed request 13/21 (pages 013)
completed request 14/21 (pages 014)
completed request 18/21 (pages 018)
completed request 20/21 (pages 020)
completed request 21/21 (pages 021)
{"images": 21, "requests": 21, "elapsed_seconds": 94.6553, "seconds_per_image": 4.5074, "completion_tokens": 89921, "end_to_end_completion_tokens_per_second": 949.984}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 12564
{
  "c

## 2200px gray220

In [14]:
show_gray_audit(2200)
gray_2200 = run_ocr("gray220 2200px", "inputs/bot_credit_bureau_2559/resolution_sweep/gray220_2200", "results/pathumma_bot_credit_bureau_21p_gray220_px2200_resolution_sweep_20260829.json")
show_all_pages(gray_2200)


[
  {
    "page": 1,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-01.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_2200/page-01.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_2200/page-01.png",
    "dimensions": [
      1555,
      2200
    ],
    "threshold": 220,
    "pixels_replaced": 3302179,
    "pixels_total": 3421000,
    "pixels_replaced_percent": 96.5267,
    "raw_sha256": "289cc186f8272818eebe04ab7817c751400dc99e89e853689e300a2a35c950b1",
    "gray220_sha256": "aca77e3bbbc84f3448de60b79c196caa439ce1ccdefeb9725d48733840f59327"
  },
  {
    "page": 2,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-02.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_2200/page-02.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_2200/page-02.png",
    "dimensions": [
      1555,
      2200
    ],
    "threshold": 220,
   

completed request 1/21 (pages 001)
completed request 5/21 (pages 005)
completed request 4/21 (pages 004)
completed request 3/21 (pages 003)
completed request 6/21 (pages 006)
completed request 2/21 (pages 002)
completed request 7/21 (pages 007)
completed request 9/21 (pages 009)
completed request 8/21 (pages 008)
completed request 10/21 (pages 010)
completed request 14/21 (pages 014)
completed request 13/21 (pages 013)
completed request 11/21 (pages 011)
completed request 12/21 (pages 012)
completed request 21/21 (pages 021)
completed request 18/21 (pages 018)
completed request 16/21 (pages 016)
completed request 15/21 (pages 015)
completed request 17/21 (pages 017)
completed request 20/21 (pages 020)
completed request 19/21 (pages 019)
{"images": 21, "requests": 21, "elapsed_seconds": 23.6417, "seconds_per_image": 1.1258, "completion_tokens": 21106, "end_to_end_completion_tokens_per_second": 892.745}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 12399
{
  "c

## Golden score for every size and preprocessing variant

In [15]:
score_path = ROOT / "results/pathumma_resolution_sweep_raw_gray220_20260829.score.json"
score_command = [str(AUDIT_PYTHON), "benchmark/score_golden_ocr.py", str(GOLDEN), str(score_path), "--target", "raw_px1024=results/pathumma_bot_credit_bureau_21p_raw_px1024_resolution_sweep_20260829.json", "--target", "gray220_px1024=results/pathumma_bot_credit_bureau_21p_gray220_px1024_resolution_sweep_20260829.json", "--target", "raw_px1400=results/pathumma_bot_credit_bureau_21p_raw_px1400_resolution_sweep_20260829.json", "--target", "gray220_px1400=results/pathumma_bot_credit_bureau_21p_gray220_px1400_resolution_sweep_20260829.json", "--target", "raw_px1600=results/pathumma_bot_credit_bureau_21p_raw_px1600_resolution_sweep_20260829.json", "--target", "gray220_px1600=results/pathumma_bot_credit_bureau_21p_gray220_px1600_resolution_sweep_20260829.json", "--target", "raw_px1800=results/pathumma_bot_credit_bureau_21p_raw_px1800_resolution_sweep_20260829.json", "--target", "gray220_px1800=results/pathumma_bot_credit_bureau_21p_gray220_px1800_resolution_sweep_20260829.json", "--target", "raw_px2000=results/pathumma_bot_credit_bureau_21p_raw_px2000_resolution_sweep_20260829.json", "--target", "gray220_px2000=results/pathumma_bot_credit_bureau_21p_gray220_px2000_resolution_sweep_20260829.json", "--target", "raw_px2200=results/pathumma_bot_credit_bureau_21p_raw_px2200_resolution_sweep_20260829.json", "--target", "gray220_px2200=results/pathumma_bot_credit_bureau_21p_gray220_px2200_resolution_sweep_20260829.json"]
print("Command:", " ".join(score_command))
completed = subprocess.run(score_command, cwd=ROOT, text=True, capture_output=True, check=True)
print(completed.stdout)
if completed.stderr:
    print("STDERR:\n" + completed.stderr)
score_payload = json.loads(score_path.read_text(encoding="utf-8"))
for label, target in score_payload["targets"].items():
    summary = target["summary"]
    print("\n", label)
    print(json.dumps({
        "word_accuracy_percent": summary["words"]["accuracy_percent"],
        "word_deltas": {key: summary["words"][key] for key in ("substituted", "missing", "extra")},
        "number_accuracy_percent": summary["number_tokens"]["accuracy_percent"],
    }, ensure_ascii=False, indent=2))


Command: python benchmark/score_golden_ocr.py ./results/bot_credit_bureau_21p_golden_transcript_20260829.json ./results/pathumma_resolution_sweep_raw_gray220_20260829.score.json --target raw_px1024=results/pathumma_bot_credit_bureau_21p_raw_px1024_resolution_sweep_20260829.json --target gray220_px1024=results/pathumma_bot_credit_bureau_21p_gray220_px1024_resolution_sweep_20260829.json --target raw_px1400=results/pathumma_bot_credit_bureau_21p_raw_px1400_resolution_sweep_20260829.json --target gray220_px1400=results/pathumma_bot_credit_bureau_21p_gray220_px1400_resolution_sweep_20260829.json --target raw_px1600=results/pathumma_bot_credit_bureau_21p_raw_px1600_resolution_sweep_20260829.json --target gray220_px1600=results/pathumma_bot_credit_bureau_21p_gray220_px1600_resolution_sweep_20260829.json --target raw_px1800=results/pathumma_bot_credit_bureau_21p_raw_px1800_resolution_sweep_20260829.json --target gray220_px1800=results/pathumma_bot_credit_bureau_21p_gray220_px1800_resolution_sw

{"raw_px1024": {"scored_pages": 21, "words": {"correct": 6098, "substituted": 729, "missing": 38, "extra": 5092, "golden_count": 6865, "ocr_count": 11919, "accuracy_percent": 88.827}, "number_tokens": {"correct": 282, "substituted": 26, "missing": 56, "extra": 11, "accuracy_percent": 77.473}}, "gray220_px1024": {"scored_pages": 21, "words": {"correct": 6816, "substituted": 41, "missing": 8, "extra": 11, "golden_count": 6865, "ocr_count": 6868, "accuracy_percent": 99.286}, "number_tokens": {"correct": 348, "substituted": 11, "missing": 5, "extra": 2, "accuracy_percent": 95.604}}, "raw_px1400": {"scored_pages": 21, "words": {"correct": 6775, "substituted": 88, "missing": 2, "extra": 1719, "golden_count": 6865, "ocr_count": 8582, "accuracy_percent": 98.689}, "number_tokens": {"correct": 333, "substituted": 14, "missing": 17, "extra": 7, "accuracy_percent": 91.484}}, "gray220_px1400": {"scored_pages": 21, "words": {"correct": 6844, "substituted": 17, "missing": 4, "extra": 14, "golden_coun